# GBM Paper Follow-Up

Cohort numbers of the text, Fig 1b and Supp Fig 1a–c.

Origin: analysis/survival/GBM_MS_Summary.ipynb (Spencer Watson); only the cells that produce
paper output are kept.

- **Cohort numbers**: 273 patients per centre, 260 in the survival analyses, age range and the
  patients younger than 45, overall survival range and KM median of the survival cohort,
  number of clinical-table columns.
- **Fig 1b top** numbers ("Range in overall survival"): OS range, KM median, by centre, on
  every subject of the clinical tables (N = 287).
- **Supp Fig 1a–1b** ("Check Cox HR assumptions"): PH diagnostics and τ selection
  (N = 287; Schoenfeld p MGMT 0.2031, age 0.0147, sex 0.8606) and again on the
  260-patient survival cohort.
- **Fig 1b bottom** and **Supp Fig 1c** ("RMST Survival Modeling"): KM and adjusted RMST
  (τ = 3 y) for age, MGMT, p53, sex and centre, and their forest (MGMT +7.1, age > 65 −5.4,
  female +3.1 months), plus the same model on the survival cohort (where the text's
  "p53 p = 0.085" comes from).

Comparison with the published outputs: the N = 287 runs reproduce the published Fig 1b and
Supp Fig 1a–c numbers above.

## Load data and metadata

In [ ]:
import matplotlib.gridspec as gridspec
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
from lifelines import KaplanMeierFitter
from scipy.ndimage import uniform_filter1d

from gbmcollaboration import AllDataCenters
from gbmcollaboration.paper import cohort, output
import rpy2.robjects as ro
from rpy2.robjects import pandas2ri
from rpy2.robjects.conversion import localconverter

ro.r("suppressPackageStartupMessages({library(survival); library(survRM2)})")

In [ ]:
# Parameters
OUT = output.setup("fig1b_suppfig1_GBM_MS_Summary")
sc.set_figure_params(dpi=200, color_map="viridis_r")
MIN_AGE_STATED = 45  # Methods: "all patients were aged 45 years or older"

# =============================================================================
# CONFIG
# =============================================================================

TIME_COL  = "os_years"
EVENT_COL = "os_event"
TAU       = 3.0

FIG_WIDTH  = 4.5
FIG_HEIGHT = 3.0
GROUP_COLORS_2  = ["#ff7b02", "#007ebc"]           # orange / blue (binary plots)
GROUP_COLORS_3  = ["#ff7b02", "#007ebc", "#2ca02c"] # orange / blue / green (center)
TITLE_SIZE      = 10
AXIS_LABEL_SIZE = 12
TICK_SIZE       = 8
LEGEND_SIZE     = 9
KM_LINEWIDTH    = 1
CI_BAND_ALPHA   = 0.15
SPINE_WIDTH     = 0.4
TICK_WIDTH      = 0.4
TICK_LENGTH     = 1.0
CENSOR_SIZE     = 4

SAVE_DIR = OUT

# Site code mapping
SITE_LABELS = {"CH": "CHUV", "ER": "UKER", "CR": "Charité"}

In [ ]:
def ph_diagnostics(clin_table: pd.DataFrame, save_dir, suffix: str = "", panels: bool = False) -> dict:
    """Cox model of MGMT + age + sex, cox.zph on the scaled Schoenfeld residuals, PH
    diagnostic figure (Supp Fig 1a) and tau-selection figure (Supp Fig 1b) for clin_table.

    One row per "Subject Id" (baseline row first); a missing MGMT or sex counts as
    unmethylated or male. Returns n, events and the cox.zph p-values.
    """
    # =============================================================================
    # CONFIG
    # =============================================================================

    TIME_COL  = "os_years"
    EVENT_COL = "os_event"
    MGMT_COL  = "MGMT promoter methylation"
    AGE_COL   = "age_at_diagnosis"
    SEX_COL   = "Administrative gender"

    COL_METH   = "#ff7b02"
    COL_UNMETH = "#007ebc"

    SAVE_DIR = save_dir

    # =============================================================================
    # PREPARE DATA
    # =============================================================================

    clin_ph = clin_table.copy()
    clin_ph["_priority"] = clin_ph["Sample collection chronology"].map(
        {"Baseline": 0, "At recurrence": 1}).fillna(2)
    clin_ph = clin_ph.sort_values(["Subject Id", "_priority"])
    clin_ph = clin_ph.drop_duplicates(subset=["Subject Id"], keep="first")
    clin_ph = clin_ph.drop(columns=["_priority"])

    clin_ph["time"]  = pd.to_numeric(clin_ph[TIME_COL],  errors="coerce")
    clin_ph["event"] = pd.to_numeric(clin_ph[EVENT_COL], errors="coerce")
    clin_ph["mgmt"]  = (clin_ph[MGMT_COL].astype(str) == "Methylated").astype(int)
    clin_ph["sex"]   = (clin_ph[SEX_COL].astype(str) == "Female").astype(int)
    age_vals         = pd.to_numeric(clin_ph[AGE_COL], errors="coerce")
    clin_ph["age_z"] = (age_vals - age_vals.mean()) / age_vals.std()

    df = clin_ph[["time", "event", "mgmt", "age_z", "sex"]].dropna()
    df = df[df["time"] > 0].copy()

    n_meth   = int(df["mgmt"].sum())
    n_unmeth = int((df["mgmt"] == 0).sum())
    median_os = float(df[df["event"] == 1]["time"].median())

    print(f"N = {len(df)},  events = {int(df['event'].sum())}")
    print(f"MGMT methylated: {n_meth} / {len(df)}")
    print(f"Median OS (events): {median_os:.2f} years")

    with localconverter(ro.default_converter + pandas2ri.converter):
        ro.globalenv["surv_data"] = ro.conversion.py2rpy(df)

    # =============================================================================
    # COX + SCHOENFELD (already run, re-run cleanly here)
    # =============================================================================

    ro.r("""
    cox_fit  <- coxph(Surv(time, event) ~ mgmt + age_z + sex,
                      data = surv_data, ties = "efron")
    zph_test <- cox.zph(cox_fit)
    zph_id   <- cox.zph(cox_fit, transform = "identity")

    # KM curves
    km_meth   <- survfit(Surv(time, event) ~ 1,
                         data = surv_data[surv_data$mgmt == 1,])
    km_unmeth <- survfit(Surv(time, event) ~ 1,
                         data = surv_data[surv_data$mgmt == 0,])
    km_all    <- survfit(Surv(time, event) ~ 1, data = surv_data)

    t_meth   <- km_meth$time;   s_meth   <- km_meth$surv
    t_unmeth <- km_unmeth$time; s_unmeth <- km_unmeth$surv
    t_all    <- km_all$time
    h_all    <- km_all$n.event / (km_all$n.risk *
                 c(diff(c(0, km_all$time))))

    # Schoenfeld residuals
    scho_t    <- zph_id$time
    scho_mgmt <- zph_id$y[, "mgmt"]
    scho_age  <- zph_id$y[, "age_z"]
    beta_mgmt <- zph_id$y[, "mgmt"]

    lo_mgmt  <- lowess(scho_t, scho_mgmt, f = 0.5)
    lo_age   <- lowess(scho_t, scho_age,  f = 0.5)
    lo_beta  <- lowess(scho_t, beta_mgmt, f = 0.5)

    ref_beta_mgmt <- coef(cox_fit)["mgmt"]
    ref_beta_age  <- coef(cox_fit)["age_z"]

    # O/E ratio — compute manually per group
    # Methylated group
    km_log_meth   <- survfit(Surv(time, event) ~ 1,
                              data = surv_data[surv_data$mgmt == 1,])
    km_log_unmeth <- survfit(Surv(time, event) ~ 1,
                              data = surv_data[surv_data$mgmt == 0,])

    # Cumulative observed events
    cum_obs_meth   <- cumsum(km_log_meth$n.event)
    cum_obs_unmeth <- cumsum(km_log_unmeth$n.event)

    # Expected under PH: scale by overall event rate
    total_events <- sum(surv_data$event)
    total_n      <- nrow(surv_data)
    prop_meth    <- sum(surv_data$mgmt) / total_n
    prop_unmeth  <- 1 - prop_meth

    # Cumulative expected (simple proportional)
    cum_exp_meth   <- (1:length(cum_obs_meth))   * prop_meth   * 0 +
                       cumsum(km_log_meth$n.event   / km_log_meth$n.risk   *
                              km_log_meth$n.risk)
    cum_exp_unmeth <- cumsum(km_log_unmeth$n.event / km_log_unmeth$n.risk *
                              km_log_unmeth$n.risk)

    # Use log-rank expected counts approach via survdiff
    sd_mgmt <- survdiff(Surv(time, event) ~ mgmt, data = surv_data)
    """)

    mgmt_zph_p   = float(ro.r("zph_test$table['mgmt',   'p']")[0])
    age_zph_p    = float(ro.r("zph_test$table['age_z',  'p']")[0])
    sex_zph_p    = float(ro.r("zph_test$table['sex',    'p']")[0])
    global_zph_p = float(ro.r("zph_test$table['GLOBAL', 'p']")[0])

    t_meth   = np.array(ro.r("t_meth"));   s_meth   = np.array(ro.r("s_meth"))
    t_unmeth = np.array(ro.r("t_unmeth")); s_unmeth = np.array(ro.r("s_unmeth"))
    t_all    = np.array(ro.r("t_all"));    h_all    = np.array(ro.r("h_all"))
    scho_t   = np.array(ro.r("scho_t"))
    scho_mgmt = np.array(ro.r("scho_mgmt"))
    scho_age  = np.array(ro.r("scho_age"))
    beta_mgmt = np.array(ro.r("beta_mgmt"))
    lo_sx_m  = np.array(ro.r("lo_mgmt$x")); lo_sy_m = np.array(ro.r("lo_mgmt$y"))
    lo_sx_a  = np.array(ro.r("lo_age$x"));  lo_sy_a = np.array(ro.r("lo_age$y"))
    lo_tx    = np.array(ro.r("lo_beta$x")); lo_ty   = np.array(ro.r("lo_beta$y"))
    ref_hr_mgmt = float(np.exp(ro.r("ref_beta_mgmt")[0]))
    ref_hr_age  = float(np.exp(ro.r("ref_beta_age")[0]))

    # O/E via survdiff — extract observed and expected
    ro.r("""
    oe_obs_vals <- sd_mgmt$obs
    oe_exp_vals <- sd_mgmt$exp
    oe_labels   <- names(sd_mgmt$obs)
    """)
    oe_obs_vals = np.array(ro.r("oe_obs_vals"))
    oe_exp_vals = np.array(ro.r("oe_exp_vals"))

    print(f"\nOverall O/E (MGMT):")
    print(f"  Methylated:   O={oe_obs_vals[1]:.0f}, E={oe_exp_vals[1]:.1f}, O/E={oe_obs_vals[1]/oe_exp_vals[1]:.2f}")
    print(f"  Unmethylated: O={oe_obs_vals[0]:.0f}, E={oe_exp_vals[0]:.1f}, O/E={oe_obs_vals[0]/oe_exp_vals[0]:.2f}")

    # For time-varying O/E, compute from KM directly
    ro.r("""
    # Compute time-varying OE using counting process approach
    # Split follow-up into intervals and compute O/E per interval
    breaks  <- quantile(surv_data$time[surv_data$event==1],
                        probs = seq(0, 1, by = 0.1))
    breaks  <- unique(c(0, breaks))
    n_int   <- length(breaks) - 1

    oe_time  <- numeric(n_int)
    oe_ratio_meth   <- numeric(n_int)
    oe_ratio_unmeth <- numeric(n_int)

    for (i in 1:n_int) {
        t_lo <- breaks[i]
        t_hi <- breaks[i + 1]
        oe_time[i] <- (t_lo + t_hi) / 2

        sub <- surv_data[surv_data$time >= t_lo,]
        sub$time2  <- pmin(sub$time, t_hi)
        sub$event2 <- sub$event * (sub$time <= t_hi)

        if (sum(sub$event2) < 2) next

        sd_int <- tryCatch(
            survdiff(Surv(time2, event2) ~ mgmt, data = sub),
            error = function(e) NULL
        )
        if (!is.null(sd_int) && all(sd_int$exp > 0)) {
            oe_ratio_meth[i]   <- sd_int$obs[2] / sd_int$exp[2]
            oe_ratio_unmeth[i] <- sd_int$obs[1] / sd_int$exp[1]
        } else {
            oe_ratio_meth[i]   <- NA
            oe_ratio_unmeth[i] <- NA
        }
    }
    """)

    oe_time         = np.array(ro.r("oe_time"))
    oe_ratio_meth   = np.array(ro.r("oe_ratio_meth"), dtype=float)
    oe_ratio_unmeth = np.array(ro.r("oe_ratio_unmeth"), dtype=float)

    # Replace 0 with NaN (unfilled intervals)
    oe_ratio_meth[oe_ratio_meth == 0]     = np.nan
    oe_ratio_unmeth[oe_ratio_unmeth == 0] = np.nan

    # =============================================================================
    # FIGURE 1 — PH DIAGNOSTICS
    # =============================================================================

    fig1 = plt.figure(figsize=(14, 11))
    gs   = gridspec.GridSpec(2, 2, hspace=0.42, wspace=0.35)
    ax1  = fig1.add_subplot(gs[0, 0])
    ax2  = fig1.add_subplot(gs[0, 1])
    ax3  = fig1.add_subplot(gs[1, 0])
    ax4  = fig1.add_subplot(gs[1, 1])

    # Panel 1: Log-log plot
    mask_m = (s_meth   > 0) & (s_meth   < 1) & (t_meth   > 0)
    mask_u = (s_unmeth > 0) & (s_unmeth < 1) & (t_unmeth > 0)
    ax1.plot(np.log(t_meth[mask_m]),   np.log(-np.log(s_meth[mask_m])),
             color=COL_METH,   linewidth=1.5, label=f"Meth. (n={n_meth})")
    ax1.plot(np.log(t_unmeth[mask_u]), np.log(-np.log(s_unmeth[mask_u])),
             color=COL_UNMETH, linewidth=1.5, label=f"Unmeth. (n={n_unmeth})")
    ax1.set_xlabel("log(time)", fontsize=10)
    ax1.set_ylabel("log(−log(S(t)))", fontsize=10)
    ax1.set_title("Log-log survival plot (MGMT)\nParallel lines → PH satisfied",
                  fontsize=10, fontweight="bold")
    ax1.legend(fontsize=9, frameon=False)
    ax1.axhline(0, color="gray", linewidth=0.5, linestyle="--", alpha=0.5)
    ax1.spines["top"].set_visible(False)
    ax1.spines["right"].set_visible(False)

    # Panel 2: Scaled Schoenfeld residuals — show both MGMT and Age
    ax2.scatter(scho_t, scho_age, color="gray", alpha=0.35, s=12, zorder=2,
                label="Age residuals")
    ax2.plot(lo_sx_a, lo_sy_a, color="#C0392B", linewidth=2.0, zorder=3,
             label=f"Age smooth (p={age_zph_p:.3f})")
    ax2.scatter(scho_t, scho_mgmt, color="lightblue", alpha=0.35, s=12, zorder=2)
    ax2.plot(lo_sx_m, lo_sy_m, color=COL_UNMETH, linewidth=2.0, zorder=3,
             linestyle="--", label=f"MGMT smooth (p={mgmt_zph_p:.3f})")
    ax2.axhline(0, color="black", linewidth=0.8, linestyle="--", alpha=0.7)
    ax2.set_xlabel("Time (years)", fontsize=10)
    ax2.set_ylabel("Scaled Schoenfeld residual", fontsize=10)
    ax2.set_title("Scaled Schoenfeld residuals\nTime trend → PH violated",
                  fontsize=10, fontweight="bold")
    ax2.legend(fontsize=8, frameon=False)
    ax2.spines["top"].set_visible(False)
    ax2.spines["right"].set_visible(False)

    # Panel 3: Time-varying HR (MGMT)
    ax3.scatter(scho_t, np.exp(beta_mgmt), color="gray", alpha=0.4, s=15, zorder=2)
    ax3.plot(lo_tx, np.exp(lo_ty), color="#C0392B", linewidth=2.0, zorder=3,
             label="Time-varying HR (lowess)")
    ax3.axhline(ref_hr_mgmt, color="#2C3E50", linewidth=1.2, linestyle="--",
                label=f"Cox HR = {ref_hr_mgmt:.2f} (constant)")
    ax3.axhline(1.0, color="black", linewidth=0.6, linestyle=":", alpha=0.5,
                label="HR = 1")
    ax3.set_xlabel("Time (years)", fontsize=10)
    ax3.set_ylabel("Hazard ratio (meth. vs unmeth.)", fontsize=10)
    ax3.set_title("Time-varying hazard ratio (MGMT)\nFlat line → PH satisfied",
                  fontsize=10, fontweight="bold")
    ax3.legend(fontsize=9, frameon=False)
    ax3.set_ylim(bottom=0)
    ax3.spines["top"].set_visible(False)
    ax3.spines["right"].set_visible(False)

    # Panel 4: Time-varying O/E ratio
    valid_m = ~np.isnan(oe_ratio_meth)
    valid_u = ~np.isnan(oe_ratio_unmeth)
    ax4.plot(oe_time[valid_m], oe_ratio_meth[valid_m],
             color=COL_METH,   linewidth=1.5, marker="o", markersize=4,
             label=f"Meth. (n={n_meth})")
    ax4.plot(oe_time[valid_u], oe_ratio_unmeth[valid_u],
             color=COL_UNMETH, linewidth=1.5, marker="o", markersize=4,
             label=f"Unmeth. (n={n_unmeth})")
    ax4.axhline(1.0, color="black", linewidth=0.8, linestyle="--", alpha=0.7,
                label="O/E = 1 (PH satisfied)")
    ax4.set_xlabel("Time (years)", fontsize=10)
    ax4.set_ylabel("O/E ratio (per interval)", fontsize=10)
    ax4.set_title("Observed / Expected events over time\nConstant ratio → PH satisfied",
                  fontsize=10, fontweight="bold")
    ax4.legend(fontsize=9, frameon=False)
    ax4.spines["top"].set_visible(False)
    ax4.spines["right"].set_visible(False)

    fig1.suptitle(
        f"Proportional hazards diagnostics — MOSAIC GBM cohort (N={len(df)})\n"
        f"Schoenfeld p: MGMT={mgmt_zph_p:.3f}  Age={age_zph_p:.3f}  "
        f"Sex={sex_zph_p:.3f}  Global={global_zph_p:.3f}",
        fontsize=10, fontweight="bold", y=1.01)

    ph_table = pd.DataFrame({"covariate": ["MGMT", "Age", "Sex", "Global"],
                             "schoenfeld_p": [mgmt_zph_p, age_zph_p, sex_zph_p, global_zph_p],
                             "n_patients": len(df), "n_events": int(df["event"].sum())})
    output.save(SAVE_DIR, f"ph_diagnostics_cohort{suffix}", fig1, ph_table, panel="Supp Fig 1a" if panels else None)
    plt.show()
    plt.close(fig1)
    print(f"Saved ph_diagnostics_cohort{suffix}.png/.pdf")

    # =============================================================================
    # FIGURE 2 — TAU SELECTION
    # =============================================================================

    h_smooth = uniform_filter1d(h_all, size=max(1, len(t_all) // 15))

    fig2, (ax_km, ax_hz) = plt.subplots(1, 2, figsize=(12, 5))

    for t_arr, s_arr, color, label in [
        (t_meth,   s_meth,   COL_METH,   f"MGMT meth. (n={n_meth})"),
        (t_unmeth, s_unmeth, COL_UNMETH, f"MGMT unmeth. (n={n_unmeth})"),
    ]:
        t_plot = np.concatenate([[0], t_arr])
        s_plot = np.concatenate([[1], s_arr])
        ax_km.step(t_plot, s_plot, where="post",
                   color=color, linewidth=1.5, label=label)

    ax_km.axvline(1.5, color="gray",  linestyle="--", linewidth=1.0, alpha=0.8)
    ax_km.axvline(3.0, color="black", linestyle="--", linewidth=1.0, alpha=0.8)
    ax_km.text(1.55, 0.97, "τ=1.5y", fontsize=8, color="gray",  va="top")
    ax_km.text(3.05, 0.97, "τ=3y",   fontsize=8, color="black", va="top")
    ax_km.set_xlabel("Time (years)", fontsize=10)
    ax_km.set_ylabel("Survival probability", fontsize=10)
    ax_km.set_title("Kaplan-Meier by MGMT status\nwith tau markers",
                    fontsize=10, fontweight="bold")
    ax_km.set_ylim(0, 1.05)
    ax_km.set_xlim(0)
    ax_km.legend(fontsize=9, frameon=False)
    ax_km.spines["top"].set_visible(False)
    ax_km.spines["right"].set_visible(False)

    ax_hz.plot(t_all, h_smooth, color="#2C3E50", linewidth=2.0,
               label="Smoothed hazard (cohort)")
    ax_hz.axvline(1.5, color="gray",  linestyle="--", linewidth=1.0, alpha=0.8,
                  label=f"τ=1.5y (≈ median OS {median_os:.1f}y)")
    ax_hz.axvline(3.0, color="black", linestyle="--", linewidth=1.0, alpha=0.8,
                  label="τ=3y (inflection)")
    ax_hz.set_xlabel("Time (years)", fontsize=10)
    ax_hz.set_ylabel("Smoothed hazard rate", fontsize=10)
    ax_hz.set_title("Population hazard rate over time\n"
                    "Inflection and plateau justify tau selection",
                    fontsize=10, fontweight="bold")
    ax_hz.legend(fontsize=9, frameon=False)
    ax_hz.spines["top"].set_visible(False)
    ax_hz.spines["right"].set_visible(False)

    fig2.suptitle(
        f"Tau selection justification — MOSAIC GBM cohort (N={len(df)})\n"
        f"Median OS (events) = {median_os:.2f} years",
        fontsize=11, fontweight="bold")
    fig2.tight_layout()
    tau_table = pd.DataFrame({"n_patients": [len(df)], "n_events": [int(df["event"].sum())],
                              "median_os_events_years": [median_os], "tau_short_years": [1.5], "tau_long_years": [3.0]})
    output.save(SAVE_DIR, f"tau_selection_justification{suffix}", fig2, tau_table, panel="Supp Fig 1b" if panels else None)
    plt.show()
    plt.close(fig2)
    print(f"Saved tau_selection_justification{suffix}.png/.pdf")

    # =============================================================================
    # SUMMARY
    # =============================================================================

    print("\n" + "="*60)
    print("PROPORTIONAL HAZARDS ASSESSMENT SUMMARY")
    print("="*60)
    print(f"\nN={len(df)}, events={int(df['event'].sum())}, "
          f"median OS={median_os:.2f}y")
    print(f"\nSchoenfeld residuals test:")
    for var, p in [("MGMT",   mgmt_zph_p),
                   ("Age",    age_zph_p),
                   ("Sex",    sex_zph_p),
                   ("Global", global_zph_p)]:
        flag = "*** PH VIOLATED" if p < 0.05 else "(not rejected)"
        print(f"  {var:<8}: p = {p:.4f}  {flag}")
    print(f"\nConclusion:")
    if age_zph_p < 0.05 or global_zph_p < 0.05:
        print(f"  PH assumption violated for at least one covariate.")
        print(f"  RMST is the more appropriate survival estimand for this cohort.")
    else:
        print(f"  Formal Schoenfeld test does not reject PH, but visual")
        print(f"  diagnostics should be inspected for time-varying patterns.")
    print(f"\nTau selection:")
    print(f"  tau=3y  : inflection in population hazard; plateau in KM curves")
    print(f"  tau=1.5y: approximate median OS ({median_os:.2f}y), "
          f"period of maximal event accrual")

    return dict(n=len(df), events=int(df["event"].sum()), mgmt=mgmt_zph_p, age=age_zph_p,
                sex=sex_zph_p, global_=global_zph_p)


# =============================================================================
# RMST HELPER
# =============================================================================

def run_rmst(df_arm1, df_arm0, tau, cov_cols=None):
    """
    arm1 = group of interest, arm0 = reference.
    Returns dict with unadjusted and (if cov_cols) adjusted results.
    """
    df_comb = pd.concat([
        df_arm0.assign(arm=0),
        df_arm1.assign(arm=1),
    ], ignore_index=True)
    df_comb = df_comb.dropna(subset=["time", "event"])
    df_comb = df_comb[df_comb["time"] > 0]

    if tau > df_comb["time"].max():
        print(f"  Warning: tau={tau} exceeds max follow-up {df_comb['time'].max():.2f}")
        return None
    if len(df_arm1) < 5 or len(df_arm0) < 5:
        print(f"  Warning: insufficient n (arm1={len(df_arm1)}, arm0={len(df_arm0)})")
        return None

    with localconverter(ro.default_converter + pandas2ri.converter):
        r_df = ro.conversion.py2rpy(df_comb)
    ro.globalenv["surv_data"] = r_df
    ro.globalenv["tau_val"]   = float(tau)

    try:
        ro.r("""
        rmst_u <- rmst2(surv_data$time, surv_data$event, surv_data$arm, tau=tau_val)
        u_diff <- rmst_u$unadjusted.result[1, "Est."]
        u_lo   <- rmst_u$unadjusted.result[1, "lower .95"]
        u_hi   <- rmst_u$unadjusted.result[1, "upper .95"]
        u_p    <- rmst_u$unadjusted.result[1, "p"]
        rmst_h <- rmst_u$RMST.arm1$rmst["Est."]
        rmst_l <- rmst_u$RMST.arm0$rmst["Est."]
        """)

        out = {
            "n_arm1":     len(df_arm1),
            "n_arm0":     len(df_arm0),
            "tau":        tau,
            "rmst_arm1":  float(ro.r("rmst_h")[0]),
            "rmst_arm0":  float(ro.r("rmst_l")[0]),
            "unadj_diff": float(ro.r("u_diff")[0]),
            "unadj_lo":   float(ro.r("u_lo")[0]),
            "unadj_hi":   float(ro.r("u_hi")[0]),
            "unadj_p":    float(ro.r("u_p")[0]),
        }

        avail_covs = [c for c in (cov_cols or [])
                      if c in df_comb.columns and df_comb[c].notna().all()]
        if avail_covs:
            cov_r = df_comb[avail_covs].values.astype(float)
            ro.globalenv["cov_mat"] = ro.r.matrix(
                ro.FloatVector(cov_r.flatten(order="C")),
                nrow=int(cov_r.shape[0]),
                ncol=int(cov_r.shape[1]),
                byrow=True,
            )
            ro.r("""
            rmst_a <- rmst2(surv_data$time, surv_data$event, surv_data$arm,
                            tau=tau_val, covariates=cov_mat)
            a_diff <- rmst_a$adjusted.result[1, "Est."]
            a_lo   <- rmst_a$adjusted.result[1, "lower .95"]
            a_hi   <- rmst_a$adjusted.result[1, "upper .95"]
            a_p    <- rmst_a$adjusted.result[1, "p"]
            """)
            out.update({
                "adj_diff": float(ro.r("a_diff")[0]),
                "adj_lo":   float(ro.r("a_lo")[0]),
                "adj_hi":   float(ro.r("a_hi")[0]),
                "adj_p":    float(ro.r("a_p")[0]),
            })

        return out

    except Exception as e:
        print(f"  survRM2 error: {e}")
        return None

# =============================================================================
# BINARY KM + RMST PLOT
# =============================================================================

def km_rmst_plot(data, group_col, arm1_label, arm0_label,
                 title="", save_stem=None, tau=TAU, cov_cols=None):
    cols_needed = list({TIME_COL, EVENT_COL, group_col} | set(cov_cols or []))
    clean = data[cols_needed].dropna(subset=[TIME_COL, EVENT_COL, group_col]).copy()
    clean["time"]  = pd.to_numeric(clean[TIME_COL],  errors="coerce")
    clean["event"] = pd.to_numeric(clean[EVENT_COL], errors="coerce")
    clean = clean[clean["time"] > 0]

    df_arm1 = clean[clean[group_col] == arm1_label].copy()
    df_arm0 = clean[clean[group_col] == arm0_label].copy()

    res = run_rmst(df_arm1, df_arm0, tau, cov_cols=cov_cols)

    fig, ax = plt.subplots(figsize=(FIG_WIDTH, FIG_HEIGHT))

    for i, (sub, label) in enumerate([
        (df_arm0, f"{arm0_label} (n={len(df_arm0)})"),
        (df_arm1, f"{arm1_label} (n={len(df_arm1)})"),
    ]):
        kmf = KaplanMeierFitter()
        kmf.fit(sub["time"], sub["event"], label=label)
        kmf.plot_survival_function(
            ax=ax, ci_show=True,
            linewidth=KM_LINEWIDTH,
            color=GROUP_COLORS_2[i],
            ci_alpha=CI_BAND_ALPHA,
        )
        if CENSOR_SIZE > 0:
            censored = sub[sub["event"] == 0]
            surv_at_censor = kmf.survival_function_at_times(
                censored["time"]).values
            ax.plot(censored["time"], surv_at_censor, "|",
                    color=GROUP_COLORS_2[i],
                    markersize=CENSOR_SIZE,
                    markeredgewidth=TICK_WIDTH)

    ax.axvline(x=tau, color="gray", linestyle="--", linewidth=0.8, alpha=0.7)
    ax.text(tau + 0.05, 0.97, f"τ={int(tau)}y",
            fontsize=7, color="gray", va="top")

    if res:
        p_key   = "adj_p" if "adj_p" in res else "unadj_p"
        p_label = "adj p" if "adj_p" in res else "p"
        diff_mo = res["adj_diff"] * 12 if "adj_diff" in res else res["unadj_diff"] * 12
        stats_txt = (
            f"RMST diff: {diff_mo:+.1f} mo (τ={int(tau)}y)\n"
            f"{p_label} = {res[p_key]:.3f}"
        )
        ax.plot([], [], " ", label=stats_txt)

    ax.set_title(title, fontsize=TITLE_SIZE, fontweight="bold")
    ax.set_xlabel("Years", fontsize=AXIS_LABEL_SIZE)
    ax.set_ylabel("Survival probability", fontsize=AXIS_LABEL_SIZE)
    ax.tick_params(labelsize=TICK_SIZE, width=TICK_WIDTH, length=TICK_LENGTH)
    ax.set_ylim(0, 1.05)
    ax.set_xlim(0)
    ax.grid(False)
    for sp in ["top", "right"]:
        ax.spines[sp].set_visible(False)
    for sp in ["bottom", "left"]:
        ax.spines[sp].set_linewidth(SPINE_WIDTH)
    ax.legend(fontsize=LEGEND_SIZE, loc="upper right",
              framealpha=0.8, handlelength=1.2,
              borderpad=0.4, labelspacing=0.3)

    fig.tight_layout()
    if save_stem:
        output.save(SAVE_DIR, save_stem, fig, pd.DataFrame([res]) if res else None,
                    panel=f"Fig 1b {save_stem.removeprefix('km_rmst_')}")
        print(f"  Saved {save_stem}.png/.pdf")
    plt.show()
    plt.close(fig)
    return res

# =============================================================================
# THREE-WAY CENTER KM PLOT
# =============================================================================

def km_center_plot(data, tau=TAU, cov_cols=None, save_stem=None):
    """
    KM plot with all three centers. Three pairwise RMST comparisons annotated.
    Centers: CH (orange), ER (blue), CR (green).
    """
    centers    = ["CH", "ER", "CR"]
    colors     = GROUP_COLORS_3
    cov_cols   = cov_cols or []

    cols_needed = list({TIME_COL, EVENT_COL, "Center"} | set(cov_cols))
    clean = data[cols_needed].dropna(subset=[TIME_COL, EVENT_COL, "Center"]).copy()
    clean["time"]  = pd.to_numeric(clean[TIME_COL],  errors="coerce")
    clean["event"] = pd.to_numeric(clean[EVENT_COL], errors="coerce")
    clean = clean[clean["time"] > 0]

    fig, ax = plt.subplots(figsize=(FIG_WIDTH, FIG_HEIGHT))

    subsets = {}
    for code, color in zip(centers, colors):
        sub = clean[clean["Center"] == code].copy()
        subsets[code] = sub
        label = f"{SITE_LABELS[code]} (n={len(sub)})"
        kmf = KaplanMeierFitter()
        kmf.fit(sub["time"], sub["event"], label=label)
        kmf.plot_survival_function(
            ax=ax, ci_show=True,
            linewidth=KM_LINEWIDTH,
            color=color,
            ci_alpha=CI_BAND_ALPHA,
        )
        if CENSOR_SIZE > 0:
            censored = sub[sub["event"] == 0]
            surv_at_censor = kmf.survival_function_at_times(
                censored["time"]).values
            ax.plot(censored["time"], surv_at_censor, "|",
                    color=color,
                    markersize=CENSOR_SIZE,
                    markeredgewidth=TICK_WIDTH)

    ax.axvline(x=tau, color="gray", linestyle="--", linewidth=0.8, alpha=0.7)
    ax.text(tau + 0.05, 0.97, f"τ={int(tau)}y",
            fontsize=7, color="gray", va="top")

    # Pairwise RMST: CH vs ER, CH vs CR, CR vs ER
    pairs = [("CH", "ER"), ("CH", "CR"), ("CR", "ER")]
    res_center = {}
    for arm1_code, arm0_code in pairs:
        key = f"{arm1_code}-{arm0_code}"
        res_center[key] = run_rmst(
            subsets[arm1_code], subsets[arm0_code],
            tau=tau, cov_cols=cov_cols
        )

    # Annotate pairwise p-values as legend entries
    for (arm1_code, arm0_code) in pairs:
        key = f"{arm1_code}-{arm0_code}"
        res = res_center[key]
        if res:
            p_key   = "adj_p" if "adj_p" in res else "unadj_p"
            p_label = "adj p" if "adj_p" in res else "p"
            label_str = (
                f"{SITE_LABELS[arm1_code]}–{SITE_LABELS[arm0_code]} "
                f"{p_label} = {res[p_key]:.3f}"
            )
        else:
            label_str = f"{SITE_LABELS[arm1_code]}–{SITE_LABELS[arm0_code]}: insufficient data"
        ax.plot([], [], " ", label=label_str)

    ax.set_title("Collection Center", fontsize=TITLE_SIZE, fontweight="bold")
    ax.set_xlabel("Years", fontsize=AXIS_LABEL_SIZE)
    ax.set_ylabel("Survival probability", fontsize=AXIS_LABEL_SIZE)
    ax.tick_params(labelsize=TICK_SIZE, width=TICK_WIDTH, length=TICK_LENGTH)
    ax.set_ylim(0, 1.05)
    ax.set_xlim(0)
    ax.grid(False)
    for sp in ["top", "right"]:
        ax.spines[sp].set_visible(False)
    for sp in ["bottom", "left"]:
        ax.spines[sp].set_linewidth(SPINE_WIDTH)
    ax.legend(fontsize=LEGEND_SIZE, loc="upper right",
              framealpha=0.8, handlelength=1.2,
              borderpad=0.4, labelspacing=0.3)

    fig.tight_layout()
    if save_stem:
        output.save(SAVE_DIR, save_stem, fig,
                    pd.DataFrame({k: v for k, v in res_center.items() if v}).T,
                    panel=f"Fig 1b {save_stem.removeprefix('km_rmst_')}")
        print(f"  Saved {save_stem}.png/.pdf")
    plt.show()
    plt.close(fig)
    return res_center

def get_res(res, key="adj"):
    """Extract adjusted (or unadjusted fallback) diff/lo/hi/p from result dict."""
    if res is None:
        return np.nan, np.nan, np.nan, np.nan
    if f"{key}_diff" in res:
        return (res[f"{key}_diff"] * 12,
                res[f"{key}_lo"]   * 12,
                res[f"{key}_hi"]   * 12,
                res[f"{key}_p"])
    # fallback to unadjusted
    return (res["unadj_diff"] * 12,
            res["unadj_lo"]   * 12,
            res["unadj_hi"]   * 12,
            res["unadj_p"])

def fmt_p(p):
    if np.isnan(p):
        return "—"
    if p < 0.0001:
        return f"{p:.2e}"
    return f"{p:.4f}"

def y_to_frac(y_data):
    return (y_data - y_lo) / (y_hi - y_lo)

### Load clinical data

In [ ]:
# Load processed clinical tables for all centers and combine into one cohort
clin_out = AllDataCenters.load_clinical(exclude_idh1_mut=False)

print("Total rows:", len(clin_out))
print("Totals by center:\n", clin_out.groupby("center").size())

# The paper cohort (loader exclusions, one block per patient) and its survival cohort
paper_cohort = cohort.clinical()
surv = cohort.survival_cohort()

### Cohort numbers in the text

273 patients (per centre), 260 in the survival analyses (13 Charité patients with immature
follow-up excluded), age range, OS range and KM median of the survival cohort,
and the number of clinical-table columns listed in Supp Table 1c.

In [ ]:
print(f"Patients: {len(paper_cohort)} {paper_cohort['center'].value_counts().to_dict()}")
print(f"In the survival analyses: {len(surv)} (excluded for immature follow-up: "
      f"{int((~paper_cohort['survival_eligible']).sum())} Charité patients)")
with_recurrence = AllDataCenters.load_clinical(exclude_idh1_mut=True, exclude_patient_ids=True, exclude_relapse=False)
print("Rows with recurrence samples kept (the Methods' 72 / 152 / 73): "
      f"{with_recurrence['center'].value_counts().to_dict()}")

# Age
age = pd.to_numeric(paper_cohort["age_at_diagnosis"], errors="coerce")
print(f"\nAge at diagnosis: {age.min():.0f}-{age.max():.0f} years, median {age.median():.0f}")
young = paper_cohort.loc[age < MIN_AGE_STATED, ["patient", "center", "age_at_diagnosis",
                                                "violates_cohort_inclusion_criteria_cohort1", "cohort_violations"]]
print(f"Patients younger than {MIN_AGE_STATED}:\n{young.round(0).to_string()}")

# Overall survival of the survival cohort
km_all = KaplanMeierFitter().fit(surv["os_years"], surv["os_event"])
print(f"\nOverall survival, survival cohort (n = {len(surv)}): "
      f"{surv['os_years'].min() * 12:.1f}-{surv['os_years'].max() * 12:.1f} months, "
      f"KM median {km_all.median_survival_time_ * 12:.1f} months, events {int(surv['os_event'].sum())}")

# Columns of the clinical tables: Supp Table 1c lists them, block ID index included
clinical_columns = [paper_cohort.index.name, *paper_cohort.columns.drop(["patient", "survival_eligible"])]
id_columns = ["block", "Subject Id", "center", "Site code"]
print(f"\nColumns of the clinical tables: {len(clinical_columns)} (Supp Table 1c lists 255: the same "
      "plus os_days, os_days_signed, pfs_days, pfs_days_signed, which the loader no longer returns); "
      f"ID columns: {sum(c in id_columns for c in clinical_columns)}")

### Range in overall survival

In [ ]:
# One row per subject (baseline row first)
clin_dedup = clin_out.copy()
clin_dedup["_priority"] = clin_dedup["Sample collection chronology"].map(
    {"Baseline": 0, "At recurrence": 1}
).fillna(2)
clin_dedup = clin_dedup.sort_values(["Subject Id", "_priority"])
clin_dedup = clin_dedup.drop_duplicates(subset=["Subject Id"], keep="first")
clin_dedup = clin_dedup.drop(columns=["_priority"])

# Use deduplicated clinical data (one row per patient, 'a' block preferred)
clin_os = clin_dedup[clin_dedup['os_years'].notna() & clin_dedup['os_event'].notna()].copy()

print(f"N patients: {len(clin_os)}, Deaths: {int(clin_os['os_event'].sum())}, "
      f"Censored: {int((clin_os['os_event'] == 0).sum())}")
print(f"OS range: {clin_os['os_years'].min():.3f} - {clin_os['os_years'].max():.3f} years")

# Overall KM
kmf = KaplanMeierFitter()
kmf.fit(clin_os['os_years'], clin_os['os_event'], label='All')
print(f"\nMedian OS (KM): {kmf.median_survival_time_:.3f} years")

# By center
print("\nBy center:")
for center in sorted(clin_os['center'].unique()):
    sub = clin_os[clin_os['center'] == center]
    kmf_c = KaplanMeierFitter()
    kmf_c.fit(sub['os_years'], sub['os_event'])
    print(f"  {center:8s}  n={len(sub):3d}  events={int(sub['os_event'].sum()):3d}  median_OS={kmf_c.median_survival_time_:.3f} yr")

# Plot
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.5))

kmf.plot_survival_function(ax=ax1, ci_show=True, linewidth=2)
ax1.axhline(0.5, color='red', ls='--', lw=1, alpha=0.5)
ax1.set(xlabel='Years', ylabel='Survival probability', title=f'Overall Survival (n={len(clin_os)}, median={kmf.median_survival_time_:.2f} yr)')

for center in sorted(clin_os['center'].unique()):
    sub = clin_os[clin_os['center'] == center]
    kmf_c = KaplanMeierFitter()
    kmf_c.fit(sub['os_years'], sub['os_event'], label=f'{center} (n={len(sub)})')
    kmf_c.plot_survival_function(ax=ax2, ci_show=False, linewidth=2)

ax2.set(xlabel='Years', ylabel='Survival probability', title='OS by Center')
plt.tight_layout()
output.save(OUT, "os_overall_and_by_center", fig)
plt.show()

### Check Cox HR assumptions

In [ ]:
ph_published = ph_diagnostics(clin_out, OUT, panels=True)

#### On the survival cohort

The PH diagnostics on the 260 patients of the survival analyses.

In [ ]:
ph_surv = ph_diagnostics(surv, OUT, suffix="_survival_cohort")

## RMST Survival Modeling

In [ ]:
# =============================================================================
# PREPARE DATA
# =============================================================================

df = clin_dedup[[
    TIME_COL, EVENT_COL,
    "age_at_diagnosis",
    "MGMT promoter methylation",
    "p53",
    "Administrative gender",
    "Site code",
]].copy()

df = df[df[TIME_COL].notna() & df[EVENT_COL].notna()].copy()
df["time"]  = pd.to_numeric(df[TIME_COL],  errors="coerce")
df["event"] = pd.to_numeric(df[EVENT_COL], errors="coerce")
df = df[df["time"] > 0].copy()

# Categorical groupings
df["MGMT"]      = df["MGMT promoter methylation"].map(
                      {"Methylated": "Methylated", "Unmethylated": "Unmethylated"})
df["p53"]       = df["p53"].map({"Positive": "Positive", "Negative": "Negative"})
df["Sex"]       = df["Administrative gender"]
df["Age_group"] = pd.cut(df["age_at_diagnosis"],
                          bins=[0, 65, 200], labels=["≤65", ">65"])
df["Center"]    = df["Site code"].astype(str)

# Numeric encodings for covariate adjustment
df["mgmt_num"] = (df["MGMT"] == "Methylated").astype(float)
df["p53_num"]  = (df["p53"] == "Positive").astype(float)
df["sex_num"]  = (df["Sex"] == "Female").astype(float)
df["age_z"]    = ((df["age_at_diagnosis"] - df["age_at_diagnosis"].mean()) /
                   df["age_at_diagnosis"].std())

# Center dummies (ER as reference, drop_first drops ER)
center_dummies = pd.get_dummies(df["Center"], prefix="center", drop_first=True).astype(float)
df = pd.concat([df, center_dummies], axis=1)
center_dummy_cols = center_dummies.columns.tolist()

print(f"Patients with complete survival data: {len(df)}")
print(f"Center dummy columns (ER is reference): {center_dummy_cols}")
print(df[["MGMT", "p53", "Sex", "Age_group", "Center"]].apply(
    lambda c: c.value_counts(dropna=False)).T)

# =============================================================================
# GENERATE KM + RMST PLOTS
# =============================================================================

print("\n--- Age ---")
res_age = km_rmst_plot(
    df, "Age_group", arm1_label=">65", arm0_label="≤65",
    title="Age at Diagnosis (≤65 vs >65)",
    save_stem=f"km_rmst_age",
    cov_cols=["mgmt_num", "p53_num", "sex_num"] + center_dummy_cols,
)

print("\n--- MGMT ---")
res_mgmt = km_rmst_plot(
    df, "MGMT", arm1_label="Methylated", arm0_label="Unmethylated",
    title="MGMT Promoter Methylation",
    save_stem=f"km_rmst_mgmt",
    cov_cols=["p53_num", "sex_num", "age_z"] + center_dummy_cols,
)

print("\n--- p53 ---")
res_p53 = km_rmst_plot(
    df, "p53", arm1_label="Positive", arm0_label="Negative",
    title="p53 Status",
    save_stem=f"km_rmst_p53",
    cov_cols=["mgmt_num", "sex_num", "age_z"] + center_dummy_cols,
)

print("\n--- Sex ---")
res_sex = km_rmst_plot(
    df, "Sex", arm1_label="Female", arm0_label="Male",
    title="Sex",
    save_stem=f"km_rmst_sex",
    cov_cols=["mgmt_num", "p53_num", "age_z"] + center_dummy_cols,
)

print("\n--- Center ---")
res_center = km_center_plot(
    df, tau=TAU,
    cov_cols=["mgmt_num", "p53_num", "sex_num", "age_z"],
    save_stem=f"km_rmst_center",
)

# =============================================================================
# FOREST PLOT (SUPPLEMENTARY)
# Uses matplotlib table for clean text layout — no text positioning arithmetic
# =============================================================================

forest_rows = [
    ("MGMT methylated vs unmethylated", res_mgmt,
     res_mgmt["n_arm1"] if res_mgmt else np.nan,
     res_mgmt["n_arm0"] if res_mgmt else np.nan),
    ("Age >65 vs ≤65", res_age,
     res_age["n_arm1"]  if res_age  else np.nan,
     res_age["n_arm0"]  if res_age  else np.nan),
    ("Sex: Female vs Male", res_sex,
     res_sex["n_arm1"]  if res_sex  else np.nan,
     res_sex["n_arm0"]  if res_sex  else np.nan),
    ("p53 positive vs negative", res_p53,
     res_p53["n_arm1"]  if res_p53  else np.nan,
     res_p53["n_arm0"]  if res_p53  else np.nan),
]

# Unpack center pairwise results into separate rows
center_pairs = [
    ("Center: CHUV vs UKER",    res_center.get("CH-ER")),
    ("Center: CHUV vs Charité", res_center.get("CH-CR")),
    ("Center: Charité vs UKER", res_center.get("CR-ER")),
]
for label, res in center_pairs:
    n1 = res["n_arm1"] if res else np.nan
    n0 = res["n_arm0"] if res else np.nan
    forest_rows.append((label, res, n1, n0))

# Build records
records = []
for label, res, n1, n0 in forest_rows:
    diff, lo, hi, p = get_res(res)
    records.append({
        "label":   label,
        "n_comp":  int(n1) if not np.isnan(n1) else "—",
        "n_ref":   int(n0) if not np.isnan(n0) else "—",
        "diff_mo": diff,
        "lo_mo":   lo,
        "hi_mo":   hi,
        "p":       p,
    })

forest_df = pd.DataFrame(records)

# =============================================================================
# SUMMARY TABLE
# =============================================================================

print("\n" + "="*80)
print(f"RMST SUMMARY (τ={int(TAU)} years, adjusted for all other covariates + center)")
print("="*80)
print(f"{'Covariate':<40} {'n comp/ref':<12} {'Diff (mo)':>10} "
      f"{'95% CI':>22} {'adj p':>8}")
print("-"*80)
for _, row in forest_df.iterrows():
    n_str  = f"{row['n_comp']}/{row['n_ref']}"
    if not np.isnan(row["diff_mo"]):
        ci_str = f"[{row['lo_mo']:+.1f}, {row['hi_mo']:+.1f}]"
        p_str  = f"{row['p']:.3f}" + (" *" if row["p"] < 0.05 else "")
        d_str  = f"{row['diff_mo']:+.1f}"
    else:
        ci_str = "—"
        p_str  = "—"
        d_str  = "—"
    print(f"{row['label']:<40} {n_str:<12} {d_str:>10} {ci_str:>22} {p_str:>8}")

In [ ]:
# FOREST PLOT
# Requires: forest_df already from previous cell, TAU defined


# Aesthetics
FP_FIG_W      = 15       # total figure width (inches)
FP_FIG_H      = 5      # total figure height (inches)
FP_DPI        = 300

FP_FS_HDR     = 8.5      # header font size
FP_FS_ROW     = 8.5      # row text font size
FP_FS_AXIS    = 8.5      # x-axis label font size
FP_FS_TITLE   = 9.5      # title font size

FP_MARKER     = "D"      # marker style
FP_MARKER_SZ  = 5
FP_CAP_SZ     = 3
FP_LW         = 1.3      # error bar line width
FP_ELW        = 1.1      # error bar cap line width

FP_COL_SIG    = "#C0392B"   # color for significant rows
FP_COL_NS     = "#2C3E50"   # color for non-significant rows
FP_COL_N      = "gray"      # color for n column

FP_ROW_H      = 0.55     # row height in inches (controls y spacing)
FP_PLOT_FRAC  = 0.42     # fraction of figure width used by the error bar plot area
                          # remaining fraction is split: labels left, stats right

FP_VLINE_COL  = "#7F8C8D"
FP_GRID_ALPHA = 0.4

FP_SAVE_STEM  = f"{SAVE_DIR}/forest_rmst_clinical_covariates"

# Data (edit labels here)
fp_rows = [
    ("MGMT: methylated vs unmethylated",  res_mgmt,
     res_mgmt["n_arm1"] if res_mgmt else np.nan,
     res_mgmt["n_arm0"] if res_mgmt else np.nan),

    ("Age: >65 vs ≤65",                   res_age,
     res_age["n_arm1"]  if res_age  else np.nan,
     res_age["n_arm0"]  if res_age  else np.nan),

    ("Sex: female vs male",               res_sex,
     res_sex["n_arm1"]  if res_sex  else np.nan,
     res_sex["n_arm0"]  if res_sex  else np.nan),

    ("p53: positive vs negative",         res_p53,
     res_p53["n_arm1"]  if res_p53  else np.nan,
     res_p53["n_arm0"]  if res_p53  else np.nan),

    ("Center: CHUV vs UKER",              res_center.get("CH-ER"),
     res_center["CH-ER"]["n_arm1"] if res_center.get("CH-ER") else np.nan,
     res_center["CH-ER"]["n_arm0"] if res_center.get("CH-ER") else np.nan),

    ("Center: CHUV vs Charité",           res_center.get("CH-CR"),
     res_center["CH-CR"]["n_arm1"] if res_center.get("CH-CR") else np.nan,
     res_center["CH-CR"]["n_arm0"] if res_center.get("CH-CR") else np.nan),

    ("Center: Charité vs UKER",           res_center.get("CR-ER"),
     res_center["CR-ER"]["n_arm1"] if res_center.get("CR-ER") else np.nan,
     res_center["CR-ER"]["n_arm0"] if res_center.get("CR-ER") else np.nan),
]

records = []
for label, res, n1, n0 in fp_rows:
    diff, lo, hi, p = get_res(res)
    records.append({
        "label":   label,
        "n_comp":  int(n1) if not np.isnan(n1) else "—",
        "n_ref":   int(n0) if not np.isnan(n0) else "—",
        "diff_mo": diff,
        "lo_mo":   lo,
        "hi_mo":   hi,
        "p":       p,
        "sig":     (not np.isnan(p)) and p < 0.05,
    })

fp_df = pd.DataFrame(records)
n_rows = len(fp_df)

# Layout
fig, ax = plt.subplots(figsize=(FP_FIG_W, FP_FIG_H))

# y positions: top row = highest index
y_pos = np.arange(n_rows - 1, -1, -1, dtype=float)

# Alternating row shading
for i in range(n_rows):
    if i % 2 == 0:
        ax.axhspan(y_pos[i] - 0.45, y_pos[i] + 0.45,
                   color="whitesmoke", zorder=0)

# Error bars
for i, row in fp_df.iterrows():
    y   = y_pos[i]
    col = FP_COL_SIG if row["sig"] else FP_COL_NS
    if not np.isnan(row["diff_mo"]):
        ax.errorbar(
            x=row["diff_mo"], y=y,
            xerr=[[row["diff_mo"] - row["lo_mo"]],
                  [row["hi_mo"]   - row["diff_mo"]]],
            fmt=FP_MARKER, color=col,
            markersize=FP_MARKER_SZ,
            capsize=FP_CAP_SZ,
            linewidth=FP_LW,
            elinewidth=FP_ELW,
            zorder=3,
        )

# Reference line
ax.axvline(x=0, color=FP_VLINE_COL, linewidth=0.9, linestyle="--", zorder=2)

# x-axis limits
finite_lo  = fp_df["lo_mo"].dropna()
finite_hi  = fp_df["hi_mo"].dropna()
x_min_data = finite_lo.min() if len(finite_lo) else -5
x_max_data = finite_hi.max() if len(finite_hi) else 5
x_pad      = max((x_max_data - x_min_data) * 0.08, 1.0)
ax.set_xlim(x_min_data - x_pad, x_max_data + x_pad)
ax.set_ylim(-0.7, n_rows - 0.3)

# y-axis: no ticks, labels drawn as text to left (see below)
ax.set_yticks([])
ax.set_xlabel(
    f"RMST difference (months, τ={int(TAU)} years)  |  comparison vs reference",
    fontsize=FP_FS_AXIS,
)
ax.set_title(
    f"Adjusted RMST difference by clinical covariate (τ={int(TAU)} years)\n"
    "",
    fontsize=FP_FS_TITLE, pad=8,
)
for sp in ["top", "right", "left"]:
    ax.spines[sp].set_visible(False)
ax.spines["bottom"].set_linewidth(0.6)
ax.tick_params(axis="x", labelsize=FP_FS_AXIS, width=0.6, length=3)
ax.grid(axis="x", linewidth=0.4, alpha=FP_GRID_ALPHA, zorder=0)

# Text columns using transAxes
X_LABEL   = -0.52   # covariate label  (right-aligned)
X_N       = -0.02   # n column         (right-aligned)
X_EST     =  1.02   # estimate + CI    (left-aligned)
X_P       =  1.38   # p-value          (left-aligned)

trans = ax.transAxes
y_lo, y_hi = ax.get_ylim()

# Headers
header_frac = y_to_frac(n_rows - 0.05)
ax.text(X_LABEL, header_frac, "Covariate",
        transform=trans, fontsize=FP_FS_HDR, fontweight="bold",
        ha="right", va="bottom")
ax.text(X_N, header_frac, "n (comp/ref)",
        transform=trans, fontsize=FP_FS_HDR, fontweight="bold",
        ha="right", va="bottom")
ax.text(X_EST, header_frac, "Estimate [95% CI] (months)",
        transform=trans, fontsize=FP_FS_HDR, fontweight="bold",
        ha="left", va="bottom")
ax.text(X_P, header_frac, "adj p",
        transform=trans, fontsize=FP_FS_HDR, fontweight="bold",
        ha="left", va="bottom")

# Rows
for i, row in fp_df.iterrows():
    yf  = y_to_frac(y_pos[i])
    col = FP_COL_SIG if row["sig"] else FP_COL_NS

    # Covariate label
    ax.text(X_LABEL, yf, row["label"],
            transform=trans, fontsize=FP_FS_ROW,
            ha="right", va="center", color=FP_COL_NS)

    # n
    n_str = f"{row['n_comp']}/{row['n_ref']}"
    ax.text(X_N, yf, n_str,
            transform=trans, fontsize=FP_FS_ROW,
            ha="right", va="center", color=FP_COL_N)

    # Estimate [95% CI]
    if not np.isnan(row["diff_mo"]):
        est_str = (f"{row['diff_mo']:+.1f} "
                   f"[{row['lo_mo']:+.1f}, {row['hi_mo']:+.1f}]")
        p_str   = fmt_p(row["p"]) + (" *" if row["sig"] else "")
    else:
        est_str = "—"
        p_str   = "—"

    ax.text(X_EST, yf, est_str,
            transform=trans, fontsize=FP_FS_ROW,
            ha="left", va="center", color=col)
    ax.text(X_P, yf, p_str,
            transform=trans, fontsize=FP_FS_ROW,
            ha="left", va="center", color=col)

# Adjust margins
fig.subplots_adjust(left=0.38, right=0.72)

output.save(SAVE_DIR, "forest_rmst_clinical_covariates", fig, forest_df, panel="Supp Fig 1c")
plt.show()
plt.close(fig)
print(f"Saved {FP_SAVE_STEM}.png/.pdf")

### Fig 1b numbers on the survival cohort

The text quotes +7.1 (MGMT), −5.4 (age > 65) and +3.1 months (female) from the run above
on every subject of the clinical tables (N = 287), but p53's "p = 0.085 at 3 years"
matches the same model on the 260-patient survival cohort. The RMST model above is re-run here
on that cohort (covariates as coded above).

In [ ]:
df_published = df
df = df_published[clin_dedup.loc[df_published.index, "Subject Id"].isin(surv["patient"]).to_numpy()].copy()
print(f"Survival cohort: {len(df)} patients")
for label, group_col, arm1, arm0, covs in (
    ("MGMT methylated vs unmethylated", "MGMT", "Methylated", "Unmethylated", ["p53_num", "sex_num", "age_z"]),
    ("Age >65 vs <=65", "Age_group", ">65", "≤65", ["mgmt_num", "p53_num", "sex_num"]),
    ("Sex: Female vs Male", "Sex", "Female", "Male", ["mgmt_num", "p53_num", "age_z"]),
    ("p53 positive vs negative", "p53", "Positive", "Negative", ["mgmt_num", "sex_num", "age_z"]),
):
    r = run_rmst(df[df[group_col] == arm1], df[df[group_col] == arm0], TAU, covs + center_dummy_cols)
    print(f"  {label:<34} {r['n_arm1']}/{r['n_arm0']}  {r['adj_diff'] * 12:+.1f} mo "
          f"[{r['adj_lo'] * 12:+.1f}, {r['adj_hi'] * 12:+.1f}]  p = {r['adj_p']:.4f}")
df = df_published

### Versions

In [ ]:
output.versions(["numpy", "pandas", "scipy", "lifelines", "matplotlib", "rpy2"], ["survival", "survRM2"])